In [1]:
import sys
from pathlib import Path

# /app is the Docker WORKDIR; when running locally, use the repo root instead.
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "utils" / "spark_session.py").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# Load .env when running locally in Cursor (Docker injects env vars via docker-compose env_file).
from dotenv import load_dotenv

load_dotenv(PROJECT_ROOT / ".env")

import os

import yaml
import pyspark.sql.functions as F

from utils.spark_session import create_spark_session

with open(PROJECT_ROOT / "schema.yaml") as f:
    schema = yaml.safe_load(f)

BRONZE_PATH = schema["bronze"]["path"]
LEGAL_BRONZE = f"{BRONZE_PATH}/{schema['bronze']['tables']['legal_docs_raw']['path']}"
WIKI_BRONZE = f"{BRONZE_PATH}/{schema['bronze']['tables']['wiki_docs_raw']['path']}"

print(f"Project root: {PROJECT_ROOT}")
print(f"Legal docs path: {LEGAL_BRONZE}")
print(f"Wiki  docs path: {WIKI_BRONZE}")

# Give Spark more heap for large text tokenization in Docker.
os.environ.setdefault(
    "PYSPARK_SUBMIT_ARGS",
    "--driver-memory 4g --executor-memory 4g pyspark-shell",
)

spark = create_spark_session("explore-bronze")
print(f"Spark version: {spark.version}")

Project root: /app
Legal docs path: s3a://cs611-project/bronze/legal_docs_raw
Wiki  docs path: s3a://cs611-project/bronze/wiki_docs_raw
:: loading settings :: url = jar:file:/usr/local/lib/python3.12/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /root/.ivy2/cache
The jars for the packages stored in: /root/.ivy2/jars
io.delta#delta-spark_2.12 added as a dependency
org.apache.hadoop#hadoop-aws added as a dependency
com.amazonaws#aws-java-sdk-bundle added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-9d6bbef7-13bd-434d-be11-bbb1980317b3;1.0
	confs: [default]
	found io.delta#delta-spark_2.12;3.2.1 in central
	found io.delta#delta-storage;3.2.1 in central
	found org.antlr#antlr4-runtime;4.9.3 in central
	found org.apache.hadoop#hadoop-aws;3.3.4 in central
	found com.amazonaws#aws-java-sdk-bundle;1.12.262 in central
	found org.wildfly.openssl#wildfly-openssl;1.0.7.Final in central
:: resolution report :: resolve 235ms :: artifacts dl 9ms
	:: modules in use:
	com.amazonaws#aws-java-sdk-bundle;1.12.262 from central in [default]
	io.delta#delta-spark_2.12;3.2.1 from central in [default]
	io.delta#delta-storage;3.2.1 from central in [default]
	org.antlr#antlr4-runtime;4.9.3 f

Spark version: 3.5.4


## R2 folder structure

Browse the medallion layers on R2 using Spark's S3A filesystem (same credentials as the pipeline). Run the setup cell above first.

In [ ]:
BUCKET_ROOT = f"s3a://{schema['r2']['bucket']}"


def list_r2_tree(spark, path: str, max_depth: int = 2, max_entries: int = 30) -> None:
    """Print folders/files under an s3a:// path (R2)."""
    sc = spark.sparkContext
    URI = sc._jvm.java.net.URI
    Path = sc._jvm.org.apache.hadoop.fs.Path
    FileSystem = sc._jvm.org.apache.hadoop.fs.FileSystem

    fs = FileSystem.get(URI(path), sc._jsc.hadoopConfiguration())

    def _walk(current: str, depth: int) -> None:
        p = Path(current)
        if not fs.exists(p):
            print(f"{'  ' * depth}(not found) {current}")
            return

        statuses = list(fs.listStatus(p))
        statuses.sort(key=lambda s: (not s.isDirectory(), s.getPath().getName().lower()))

        shown = 0
        for status in statuses:
            if shown >= max_entries:
                remaining = len(statuses) - shown
                print(f"{'  ' * (depth + 1)}... and {remaining} more")
                break

            name = status.getPath().getName()
            full = status.getPath().toString()
            prefix = "  " * depth

            if status.isDirectory():
                print(f"{prefix}{name}/")
                if depth < max_depth:
                    _walk(full, depth + 1)
            else:
                size_mb = status.getLen() / (1024 * 1024)
                print(f"{prefix}{name}  ({size_mb:.2f} MB)")

            shown += 1

    print(path)
    _walk(path, 0)


# Medallion roots from schema.yaml
LAYER_PATHS = {
    "landing": schema["landing"]["path"],
    "bronze": schema["bronze"]["path"],
    "silver": schema["silver"]["path"],
    "gold": schema["gold"]["path"],
}

print(f"Bucket root: {BUCKET_ROOT}\n")
for layer, path in LAYER_PATHS.items():
    print("=" * 70)
    print(layer)
    print("=" * 70)
    list_r2_tree(spark, path, max_depth=2, max_entries=25)
    print()

# Drill into a specific table (change path / depth as needed)
# list_r2_tree(spark, f"{schema['gold']['path']}/features_log_tfidf_svd", max_depth=2)

## Bronze tables preview

Read Delta tables from R2 and print schema, row counts, and sample rows.

In [ ]:
def preview_bronze_table(name: str, path: str, sample_n: int = 5) -> None:
    print("=" * 70)
    print(f"Bronze table: {name}")
    print(f"Path: {path}")
    print("=" * 70)

    df = spark.read.format("delta").load(path)

    print("\nSchema:")
    df.printSchema()

    row_count = df.count()
    print(f"Row count: {row_count:,}")

    print("\nColumns:")
    for col in df.columns:
        print(f"  - {col}")

    if "snapshot_date" in df.columns:
        print("\nSnapshot dates:")
        df.groupBy("snapshot_date").count().orderBy("snapshot_date").show(20, truncate=False)

    if row_count > 0:
        print(f"\nSample rows (first {sample_n}):")
        df.show(sample_n, truncate=80, vertical=False)
    else:
        print("\nTable is empty.")

    print()


preview_bronze_table("legal_docs_raw", LEGAL_BRONZE)
preview_bronze_table("wiki_docs_raw", WIKI_BRONZE)

Bronze table: legal_docs_raw
Path: s3a://cs611-project/bronze/legal_docs_raw



Schema:
root
 |-- CELEX: string (nullable = true)
 |-- Act_name: string (nullable = true)
 |-- Act_type: string (nullable = true)
 |-- Status: string (nullable = true)
 |-- EUROVOC: string (nullable = true)
 |-- Treaty: string (nullable = true)
 |-- Legal_basis_celex: string (nullable = true)
 |-- Authors: string (nullable = true)
 |-- Procedure_number: string (nullable = true)
 |-- Date_document: string (nullable = true)
 |-- Date_publication: string (nullable = true)
 |-- First_entry_into_force: string (nullable = true)
 |-- Temporal_status: string (nullable = true)
 |-- Act_cites: string (nullable = true)
 |-- Cites_links: string (nullable = true)
 |-- Act_ammends: string (nullable = true)
 |-- Ammends_links: string (nullable = true)
 |-- Eurlex_link: string (nullable = true)
 |-- ELI_link: string (nullable = true)
 |-- Proposal_link: string (nullable = true)
 |-- Oeil_link: string (nullable = true)
 |-- Additional_info: string (nullable = true)
 |-- act_raw_text: string (nullable 

Row count: 58,001

Columns:
  - CELEX
  - Act_name
  - Act_type
  - Status
  - EUROVOC
  - Treaty
  - Legal_basis_celex
  - Authors
  - Procedure_number
  - Date_document
  - Date_publication
  - First_entry_into_force
  - Temporal_status
  - Act_cites
  - Cites_links
  - Act_ammends
  - Ammends_links
  - Eurlex_link
  - ELI_link
  - Proposal_link
  - Oeil_link
  - Additional_info
  - act_raw_text
  - labels
  - snapshot_date

Snapshot dates:


+-------------+-----+
|snapshot_date|count|
+-------------+-----+
|1983-01-01   |4343 |
|1984-01-01   |1274 |
|1985-01-01   |1300 |
|1986-01-01   |1617 |
|1987-01-01   |1590 |
|1988-01-01   |1585 |
|1989-01-01   |1461 |
|1990-01-01   |1562 |
|1991-01-01   |1720 |
|1992-01-01   |1757 |
|1993-01-01   |2137 |
|1994-01-01   |2272 |
|1995-01-01   |3920 |
|1996-01-01   |3396 |
|1997-01-01   |3677 |
|1998-01-01   |3788 |
|1999-01-01   |3904 |
|2000-01-01   |3624 |
|2001-01-01   |3576 |
|2002-01-01   |3191 |
+-------------+-----+
only showing top 20 rows


Sample rows (first 5):


+----------+--------------------------------------------------------------------------------+------------------+------------+--------------------------------------------------------------------------------+------+----------------------+-------------------+----------------+-------------+----------------+----------------------+---------------+---------------------------------------+--------------------------------------------------------------------------------+-----------------------------------------------------+--------------------------------------------------------------------------------+--------------------------------------------------------------------+-----------------------------------------+------------------------------------------------------------------------+---------+---------------+--------------------------------------------------------------------------------+-----------------------------------------------------------------------+-------------+
|     CELEX|           

+--------+----------------------------------------------------------------------+--------------------------------+--------------------------------------------------------------------------------+
|      id|                                                                   url|                           title|                                                                            text|
+--------+----------------------------------------------------------------------+--------------------------------+--------------------------------------------------------------------------------+
|64515138|        https://en.wikipedia.org/wiki/Facebook%2C%20Inc.%20v.%20Duguid|        Facebook, Inc. v. Duguid|"Facebook, Inc. v. Duguid, 592 U.S. ___ (2021), was a United States Supreme C...|
|64597748|                    https://en.wikipedia.org/wiki/Bridget%20R.%20Cooks|                Bridget R. Cooks|"Bridget R. Cooks is an American scholar, writer, curator, and academic. She ...|
|64583266|          

## Legal docs quality checks

Checks on `legal_docs_raw`:
- missing key fields (`CELEX`, `act_raw_text`, `labels`)
- duplicate `CELEX` IDs
- label distribution
- document length distribution
- top raw tokens (before stopword removal / lemmatization)

In [3]:
import re

from pyspark.sql.types import ArrayType, StringType

ID_COL = "CELEX"
TEXT_COL = "act_raw_text"
LABEL_COL = "labels"

legal_df = spark.read.format("delta").load(LEGAL_BRONZE)
total_rows = legal_df.count()
print(f"Total bronze legal rows: {total_rows:,}\n")

# --- 1. Missing documents / missing fields ---
key_df = legal_df.select(
    ID_COL,
    TEXT_COL,
    LABEL_COL,
    (F.col(ID_COL).isNull() | (F.length(F.trim(F.col(ID_COL))) == 0)).alias("missing_id"),
    (F.col(TEXT_COL).isNull() | (F.length(F.trim(F.col(TEXT_COL))) == 0)).alias("missing_text"),
    (F.col(LABEL_COL).isNull() | (F.length(F.trim(F.col(LABEL_COL))) == 0)).alias("missing_label"),
)

missing_id = key_df.filter("missing_id").count()
missing_text = key_df.filter("missing_text").count()
missing_label = key_df.filter("missing_label").count()
missing_any = key_df.filter("missing_id OR missing_text OR missing_label").count()

print("=" * 70)
print("1) Missing field counts")
print("=" * 70)
print(f"  Missing {ID_COL}:      {missing_id:,}")
print(f"  Missing {TEXT_COL}:  {missing_text:,}")
print(f"  Missing {LABEL_COL}:       {missing_label:,}")
print(f"  Missing any key field: {missing_any:,} ({100 * missing_any / total_rows:.2f}%)")

if missing_any > 0:
    print("\nSample rows with missing key fields:")
    key_df.filter("missing_id OR missing_text OR missing_label").show(10, truncate=80)

Total bronze legal rows: 58,001



1) Missing field counts
  Missing CELEX:      0
  Missing act_raw_text:  24
  Missing labels:       10
  Missing any key field: 25 (0.04%)

Sample rows with missing key fields:


+--------------+------------+--------------------------------------------------------------------------------+----------+------------+-------------+
|         CELEX|act_raw_text|                                                                          labels|missing_id|missing_text|missing_label|
+--------------+------------+--------------------------------------------------------------------------------+----------+------------+-------------+
|    31998R1705|        NULL|"Avis juridique important|31998R1705Council Regulation (EC) No 1705/98 of 28 ...|     false|        true|        false|
|    31996R0663|        NULL|                                                                         Dumping|     false|        true|        false|
|    31995R2451|        NULL|                                                                      31993R2861|     false|        true|        false|
|    31983D0440|        NULL|"Avis juridique important|31983D044083/440/EEC: Commission Decision of 10 Aug

In [4]:
# --- 2. Duplicate documents ---
dup_df = (
    legal_df.filter(F.col(ID_COL).isNotNull() & (F.length(F.trim(F.col(ID_COL))) > 0))
    .groupBy(ID_COL)
    .count()
    .filter(F.col("count") > 1)
    .orderBy(F.desc("count"))
)

dup_celex_count = dup_df.count()
dup_extra_rows = dup_df.agg(F.sum(F.col("count") - 1).alias("extra")).first()["extra"] or 0

print("\n" + "=" * 70)
print("2) Duplicate documents (by CELEX)")
print("=" * 70)
print(f"  CELEX values with duplicates: {dup_celex_count:,}")
print(f"  Extra duplicate rows:         {dup_extra_rows:,}")

if dup_celex_count > 0:
    print("\nTop duplicate CELEX IDs:")
    dup_df.show(15, truncate=False)


2) Duplicate documents (by CELEX)
  CELEX values with duplicates: 0
  Extra duplicate rows:         0


In [5]:
# --- 3. Label distribution ---
print("\n" + "=" * 70)
print("3) Label distribution")
print("=" * 70)

label_dist = (
    legal_df.withColumn(
        LABEL_COL,
        F.when(F.col(LABEL_COL).isNull() | (F.length(F.trim(F.col(LABEL_COL))) == 0), F.lit("(missing)"))
        .otherwise(F.trim(F.col(LABEL_COL))),
    )
    .groupBy(LABEL_COL)
    .count()
    .withColumn("pct", F.round(100 * F.col("count") / total_rows, 2))
    .orderBy(F.desc("count"))
)

label_dist.show(25, truncate=False)


3) Label distribution


+--------------------------------------------------------------------------------+-----+-----+
|labels                                                                          |count|pct  |
+--------------------------------------------------------------------------------+-----+-----+
|Agriculture & Food Safety; Trade & Customs                                      |13041|22.48|
|Finance, Tax & Banking; Agriculture & Food Safety                               |3548 |6.12 |
|Agriculture & Food Safety; Trade & Customs; International & EU Law              |3337 |5.75 |
|Finance, Tax & Banking; Agriculture & Food Safety; Trade & Customs              |2794 |4.82 |
|Agriculture & Food Safety                                                       |2703 |4.66 |
|Industry, Technology & IP; Trade & Customs                                      |2226 |3.84 |
|Agriculture & Food Safety; Economic & Competition Policy                        |1177 |2.03 |
|Labour, Employment & Social; Agriculture & Food S

In [6]:
# --- 4. Document length distribution ---
length_df = legal_df.withColumn(
    "char_len",
    F.length(F.trim(F.coalesce(F.col(TEXT_COL), F.lit("")))),
).withColumn(
    "word_len",
    F.when(F.col("char_len") == 0, F.lit(0)).otherwise(F.size(F.split(F.trim(F.col(TEXT_COL)), r"\s+"))),
)

length_stats = length_df.agg(
    F.min("char_len").alias("min_chars"),
    F.max("char_len").alias("max_chars"),
    F.avg("char_len").alias("avg_chars"),
    F.expr("percentile_approx(char_len, 0.5)").alias("median_chars"),
    F.expr("percentile_approx(char_len, 0.95)").alias("p95_chars"),
    F.min("word_len").alias("min_words"),
    F.max("word_len").alias("max_words"),
    F.avg("word_len").alias("avg_words"),
    F.expr("percentile_approx(word_len, 0.5)").alias("median_words"),
).collect()[0]

print("\n" + "=" * 70)
print("4) Document length distribution")
print("=" * 70)
print(
    f"  Characters — min: {length_stats['min_chars']:,}, "
    f"median: {int(length_stats['median_chars']):,}, "
    f"avg: {length_stats['avg_chars']:,.0f}, "
    f"p95: {int(length_stats['p95_chars']):,}, "
    f"max: {length_stats['max_chars']:,}"
)
print(
    f"  Words      — min: {length_stats['min_words']:,}, "
    f"median: {int(length_stats['median_words']):,}, "
    f"avg: {length_stats['avg_words']:,.0f}, "
    f"max: {length_stats['max_words']:,}"
)

length_buckets = (
    length_df.withColumn(
        "char_bucket",
        F.when(F.col("char_len") == 0, "0 (empty)")
        .when(F.col("char_len") < 500, "1-499")
        .when(F.col("char_len") < 1000, "500-999")
        .when(F.col("char_len") < 2500, "1000-2499")
        .when(F.col("char_len") < 5000, "2500-4999")
        .when(F.col("char_len") < 10000, "5000-9999")
        .otherwise("10000+"),
    )
    .groupBy("char_bucket")
    .count()
    .withColumn("pct", F.round(100 * F.col("count") / total_rows, 2))
    .orderBy("char_bucket")
)

print("\nCharacter length buckets:")
length_buckets.show(truncate=False)


4) Document length distribution
  Characters — min: 0, median: 3,135, avg: 6,176, p95: 17,969, max: 1,595,516
  Words      — min: 0, median: 523, avg: 1,037, max: 302,790

Character length buckets:


+-----------+-----+-----+
|char_bucket|count|pct  |
+-----------+-----+-----+
|0 (empty)  |24   |0.04 |
|1-499      |3288 |5.67 |
|1000-2499  |14207|24.49|
|10000+     |6643 |11.45|
|2500-4999  |23643|40.76|
|500-999    |782  |1.35 |
|5000-9999  |9414 |16.23|
+-----------+-----+-----+



In [7]:
import re

from pyspark.sql.types import ArrayType, StringType

# --- 5. Top raw tokens ---
# Inline tokenization for Spark UDF (workers cannot import the local `utils` package).
def bronze_raw_tokenize(text: str) -> list[str]:
    return re.findall(r"[a-z0-9]+", str(text).lower())

raw_tokenize_udf = F.udf(bronze_raw_tokenize, ArrayType(StringType()))

print("\n" + "=" * 70)
print("5) Top raw tokens (lowercase alphanumeric, no stopword removal yet)")
print("=" * 70)

# Use a sample to avoid Spark OOM on all 58k full-text documents.
TOKEN_SAMPLE_FRACTION = 0.1
text_df = legal_df.filter(
    F.col(TEXT_COL).isNotNull() & (F.length(F.trim(F.col(TEXT_COL))) > 0)
)
sample_df = text_df.sample(withReplacement=False, fraction=TOKEN_SAMPLE_FRACTION, seed=42)
sample_n = sample_df.count()
print(f"Tokenizing {sample_n:,} sampled docs ({TOKEN_SAMPLE_FRACTION:.0%} of non-empty text)")

top_tokens = (
    sample_df.select(F.explode(raw_tokenize_udf(F.col(TEXT_COL))).alias("token"))
    .groupBy("token")
    .count()
    .orderBy(F.desc("count"))
    .limit(30)
)

top_tokens.show(30, truncate=False)


5) Top raw tokens (lowercase alphanumeric, no stopword removal yet)


Tokenizing 5,730 sampled docs (10% of non-empty text)


+----------+------+
|token     |count |
+----------+------+
|the       |445080|
|of        |273959|
|in        |144570|
|to        |139220|
|and       |120396|
|for       |83025 |
|1         |68377 |
|no        |65138 |
|a         |61007 |
|regulation|58321 |
|be        |58116 |
|article   |49008 |
|2         |47614 |
|by        |46315 |
|on        |45330 |
|shall     |38700 |
|or        |35890 |
|3         |34810 |
|commission|34764 |
|as        |34271 |
|this      |33052 |
|is        |32706 |
|with      |30074 |
|eec       |29648 |
|l         |28670 |
|that      |28465 |
|whereas   |27466 |
|community |26449 |
|4         |25870 |
|p         |25542 |
+----------+------+



## 6. Top tokens after preprocessing

Same sample as section 5. Two code cells mirror production `utils/text_preprocess.py`:

- **6a** — `preprocess_tokens_base`: stopword removal + lemmatization
- **6b** — `filter_token_noise`: drop 1-char letters and non-year digit tokens

In [9]:
import re

import nltk
from pyspark.sql.types import ArrayType, StringType

# Download NLTK data on the driver (local Spark workers share the same filesystem in Docker).
for resource in ("stopwords", "wordnet", "omw-1.4"):
    try:
        if resource == "stopwords":
            nltk.data.find("corpora/stopwords")
        elif resource == "wordnet":
            nltk.data.find("corpora/wordnet")
        else:
            nltk.data.find("corpora/omw-1.4")
    except LookupError:
        nltk.download(resource, quiet=True)


def bronze_preprocess_tokens_base(text: str) -> list[str]:
    """Inline copy of utils.text_preprocess.preprocess_tokens_base for Spark workers."""
    if not getattr(bronze_preprocess_tokens_base, "_ready", False):
        from nltk.corpus import stopwords
        from nltk.stem import WordNetLemmatizer

        bronze_preprocess_tokens_base._stops = set(stopwords.words("english"))
        bronze_preprocess_tokens_base._lem = WordNetLemmatizer()
        bronze_preprocess_tokens_base._ready = True

    stops = bronze_preprocess_tokens_base._stops
    lemmatizer = bronze_preprocess_tokens_base._lem
    out: list[str] = []

    for token in re.findall(r"[a-z0-9]+", str(text).lower()):
        if token in stops:
            continue
        out.append(lemmatizer.lemmatize(token))

    return out


preprocess_tokens_base_udf = F.udf(bronze_preprocess_tokens_base, ArrayType(StringType()))

print("\n" + "=" * 70)
print("6a) Top tokens after stopword removal + lemmatization")
print("=" * 70)

if "sample_df" not in globals():
    TOKEN_SAMPLE_FRACTION = 0.1
    text_df = legal_df.filter(
        F.col(TEXT_COL).isNotNull() & (F.length(F.trim(F.col(TEXT_COL))) > 0)
    )
    sample_df = text_df.sample(withReplacement=False, fraction=TOKEN_SAMPLE_FRACTION, seed=42)

print(f"Tokenizing {sample_df.count():,} sampled docs")

top_base_tokens = (
    sample_df.select(F.explode(preprocess_tokens_base_udf(F.col(TEXT_COL))).alias("token"))
    .groupBy("token")
    .count()
    .orderBy(F.desc("count"))
    .limit(30)
)

top_base_tokens.show(30, truncate=False)


6a) Top tokens after stopword removal + lemmatization


Tokenizing 5,730 sampled docs


+----------+-----+
|token     |count|
+----------+-----+
|1         |68382|
|regulation|59781|
|article   |52157|
|2         |47614|
|community |39120|
|shall     |38700|
|commission|34839|
|3         |34810|
|eec       |29648|
|l         |28671|
|whereas   |27466|
|4         |25870|
|p         |25553|
|european  |25042|
|ec        |24781|
|member    |21875|
|state     |21454|
|product   |21173|
|5         |20658|
|oj        |20629|
|10        |17745|
|regard    |16861|
|council   |15747|
|6         |15469|
|7         |14233|
|annex     |13742|
|decision  |13511|
|may       |13414|
|12        |12370|
|official  |12281|
+----------+-----+



In [10]:
def bronze_filter_token_noise(tokens: list[str]) -> list[str]:
    """Inline copy of utils.text_preprocess.filter_token_noise for Spark workers."""
    out: list[str] = []

    for token in tokens:
        if len(token) == 1 and token.isalpha():
            continue
        if token.isdigit():
            if len(token) != 4:
                continue
            out.append(token)
            continue
        out.append(token)

    return out


filter_token_noise_udf = F.udf(bronze_filter_token_noise, ArrayType(StringType()))

print("\n" + "=" * 70)
print("6b) Top tokens after noise filter (1-char letters + non-year digits removed)")
print("=" * 70)

if "sample_df" not in globals():
    TOKEN_SAMPLE_FRACTION = 0.1
    text_df = legal_df.filter(
        F.col(TEXT_COL).isNotNull() & (F.length(F.trim(F.col(TEXT_COL))) > 0)
    )
    sample_df = text_df.sample(withReplacement=False, fraction=TOKEN_SAMPLE_FRACTION, seed=42)

print(f"Tokenizing {sample_df.count():,} sampled docs")

top_filtered_tokens = (
    sample_df.select(
        F.explode(
            filter_token_noise_udf(preprocess_tokens_base_udf(F.col(TEXT_COL)))
        ).alias("token")
    )
    .groupBy("token")
    .count()
    .orderBy(F.desc("count"))
    .limit(30)
)

top_filtered_tokens.show(30, truncate=False)


6b) Top tokens after noise filter (1-char letters + non-year digits removed)


Tokenizing 5,730 sampled docs


+----------+-----+
|token     |count|
+----------+-----+
|regulation|59781|
|article   |52157|
|community |39120|
|shall     |38700|
|commission|34839|
|eec       |29648|
|whereas   |27466|
|european  |25042|
|ec        |24781|
|member    |21875|
|state     |21454|
|product   |21173|
|oj        |20629|
|regard    |16861|
|council   |15747|
|annex     |13742|
|decision  |13511|
|may       |13414|
|official  |12281|
|market    |11830|
|price     |11089|
|journal   |11044|
|measure   |10702|
|import    |10681|
|amended   |10054|
|particular|9800 |
|accordance|9220 |
|directive |9166 |
|aid       |8457 |
|period    |8426 |
+----------+-----+



----------------------------------------
Exception occurred during processing of request from ('127.0.0.1', 38520)
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/socketserver.py", line 318, in _handle_request_noblock
    self.process_request(request, client_address)
  File "/usr/local/lib/python3.12/socketserver.py", line 349, in process_request
    self.finish_request(request, client_address)
  File "/usr/local/lib/python3.12/socketserver.py", line 362, in finish_request
    self.RequestHandlerClass(request, client_address, self)
  File "/usr/local/lib/python3.12/socketserver.py", line 766, in __init__
    self.handle()
  File "/usr/local/lib/python3.12/site-packages/pyspark/accumulators.py", line 295, in handle
    poll(accum_updates)
  File "/usr/local/lib/python3.12/site-packages/pyspark/accumulators.py", line 267, in poll
    if self.rfile in r and func():
                           ^^^^^^
  File "/usr/local/lib/python3.12/site-packages/pyspark/accumulators.p